# ML-10 — Content Action Playbook

This notebook turns the validated ranking into a human-review queue. It preserves the same five honest features and client-grouped validation design, then maps high-ranked candidates to reason codes, actions, human checks, and no-go rules.


## 1. Ranked actions + reason codes

The final queue uses the best learned model from the same grouped comparison. Each row gets a model score, one primary reason code based on the visible feature pattern, and an action label for a content reviewer. Reason codes are descriptive explanations, not causal diagnoses.


In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn
import os, sys, json
from pathlib import Path
import pandas as pd, numpy as np
from google.colab import userdata
from huggingface_hub import whoami

REPO_DIR = Path('/content/Internship')
if not REPO_DIR.exists():
    !git clone https://github.com/imalik-7/Internship.git /content/Internship

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

from work.lib.capstone_pipeline import (
    FEATURES, LABEL, CONTEXT,
    connect_warehouse, build_analysis_frame, grouped_split,
    train_compare, best_model_name, feature_importance,
    ensure_output_dir
)

HF_TOKEN = userdata.get('HF_TOKEN')
print('HF account:', whoami(token=HF_TOKEN)['name'])

con = connect_warehouse(HF_TOKEN)
analysis_df = build_analysis_frame(con)

train_df, test_df = grouped_split(
    analysis_df,
    test_size=0.25,
    random_state=42
)

comparison, fitted_models, test_scores, _ = train_compare(
    train_df,
    test_df,
    random_state=42
)

BEST_MODEL = best_model_name(comparison)

queue = test_df[CONTEXT + FEATURES + [LABEL]].copy()
queue['model_score'] = test_scores[BEST_MODEL]

ctr_med = float(train_df['ctr_feature15'].median())
imp_q75 = float(train_df['impressions_feature15'].quantile(0.75))

def explain(row):
    if (
        row['impressions_feature15'] >= imp_q75
        and row['ctr_feature15'] < ctr_med
    ):
        return 'high_visibility_low_ctr', 'review_title_meta_and_intent'

    if 11 <= row['avg_position_feature15'] <= 20:
        return 'striking_distance', 'review_content_and_query_fit'

    if row['active_impression_days_feature15'] < 8:
        return 'thin_observation_window', 'monitor_before_editing'

    if (
        row['avg_position_feature15'] <= 10
        and row['ctr_feature15'] >= ctr_med
    ):
        return 'strong_search_capture', 'protect_and_monitor'

    return 'multi_signal_review', 'manual_content_review'

rec = queue.apply(explain, axis=1, result_type='expand')
queue['reason_code'] = rec[0]
queue['action_label'] = rec[1]

queue = queue.sort_values(
    'model_score',
    ascending=False
).reset_index(drop=True)

queue['rank'] = np.arange(1, len(queue) + 1)

print('Best model:', BEST_MODEL)
display(comparison.round(3))
display(
    queue[
        ['rank', 'content_hash_id', 'model_score', 'reason_code', 'action_label']
        + FEATURES
    ].head(20)
)


## 2. Intended use and limits

**User:** a content/SEO reviewer deciding what deserves attention first.

**Use:** inspect the top of the queue, check page/query context, then choose whether to refresh, improve metadata, protect, or monitor.

**Limit:** the score is based on a short March 2026 feature window and a later impression-decline proxy. It cannot explain why performance changed or guarantee an edit will improve traffic.


In [ ]:
print('Intended use: prioritize human review.')
print("Not valid for causal refresh claims, automatic rewriting/publishing, or claims about Google's algorithm.")
print('Held-out evaluation rows:', len(test_df))


## 3. Human review + the no-go list

Before acting, a person checks strategic relevance, query mix, seasonality/SERP changes, and whether the suggested action matches the page's real intent.

Never automate deletion/pruning, URL/canonical changes, high-stakes content changes, generated publishing, or causal claims from this score alone.


In [ ]:
top10 = queue.head(10).copy()

def check(row):
    if row['active_impression_days_feature15'] < 8:
        return 'Check data sufficiency; monitor before editing.'

    if row['reason_code'] == 'high_visibility_low_ctr':
        return 'Check query intent and SERP features before title/meta changes.'

    if row['reason_code'] == 'striking_distance':
        return 'Check ranking queries match the intended topic.'

    if row['reason_code'] == 'strong_search_capture':
        return 'Avoid unnecessary edits; verify a real problem first.'

    return 'Inspect seasonality, query mix, and page context before acting.'

top10['human_check'] = top10.apply(check, axis=1)

display(
    top10[
        ['rank', 'content_hash_id', 'model_score',
         'reason_code', 'action_label', 'human_check']
    ]
)


## 4. Monitoring / retrain triggers

Review monthly and retrain when fresh-holdout Precision@50 drops materially, the target/base rate shifts, feature distributions move outside March reference ranges, data-availability semantics change, or enough new months exist to replace the short March design with a stronger time-aware panel.


In [ ]:
reference = {
    'base_rate': float(test_df[LABEL].mean()),
    'impressions_median': float(train_df['impressions_feature15'].median()),
    'ctr_median': float(train_df['ctr_feature15'].median()),
    'position_median': float(train_df['avg_position_feature15'].median()),
    'precision_at_50': float(
        comparison.loc[
            comparison['method'].eq(BEST_MODEL),
            'precision_at_50'
        ].iloc[0]
    ),
}

print(json.dumps(reference, indent=2))


## 5. Exports for the paper

Exports are public-safe: no raw client names, domains, URLs, titles, or private queries. The full queue is regenerated from the notebook; a small pseudonymous sample and metric receipts support the paper.


In [ ]:
OUT = ensure_output_dir(REPO_DIR)

queue[
    ['rank', 'client_hash_id', 'content_hash_id',
     'model_score', 'reason_code', 'action_label'] + FEATURES
] .to_csv(
    OUT / 'capstone_action_queue.csv',
    index=False
)

queue[
    ['rank', 'content_hash_id', 'model_score',
     'reason_code', 'action_label']
] .head(20).to_csv(
    OUT / 'capstone_public_top20.csv',
    index=False
)

feature_importance(
    fitted_models[BEST_MODEL]
).to_csv(
    OUT / 'capstone_feature_importance.csv',
    index=False
)

with open(
    OUT / 'w07_action_playbook.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        {
            'best_model': BEST_MODEL,
            'monitoring_reference': reference,
            'reason_codes': sorted(queue['reason_code'].unique().tolist()),
            'actions': sorted(queue['action_label'].unique().tolist()),
        },
        f,
        indent=2
    )

print('Wrote paper exports to', OUT)


## Self-check

- [x] Ranked queue has a score, reason code, and action label.
- [x] Intended use and limits are explicit.
- [x] Human review happens before action.
- [x] No-go list blocks high-risk automation and causal overclaiming.
- [x] Monitoring/retrain triggers are concrete.
- [x] Public exports are pseudonymous/public-safe.
